### Copyright 2022-2026 Crown Copyright

```
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```

## Setup Sleeper Client
Set up the logger for the notebook and create a Sleeper Client connected to a Sleeper instance.

Prerequisites:
- Sleeper is already deployed
- A table exists and is partitioned
- AWS credentials and region are configured
- The sleeper Python package is installed

In [ ]:
import logging
import random
import string

from sleeper import SleeperClient, enable_logging

logging.basicConfig(
    level=logging.INFO,
    format="[NOTEBOOK] %(asctime)s %(levelname)s %(message)s",
    force=True,
)

jupyter_logger = logging.getLogger(__name__)

jupyter_logger.info("Starting notebook")

# Enable Sleeper Client logging (switch to DEBUG only when troubleshooting)
enable_logging(logging.INFO)

# TODO: Replace with your deployed values
table_name = "my-table"
instance_id = "instance-123"

sleeper_client = SleeperClient(instance_id)

## Helper Function
Used to create random strings for sample rows.

In [ ]:
def random_string(size=6, chars=string.ascii_letters + string.digits):
    """Create a random string of characters.

    :param size: length of string to generate
    :param chars: characters to choose from

    :return: random string
    """
    return "".join(random.choice(chars) for _ in range(size))

## Write Data in Batches
This example writes data to Sleeper in batches.
Data is ingested when the batch writer is closed (automatic when using a with block).

For large-scale imports, use the bulk import method.

In [ ]:
num_batches = 1

# Recommended method is to use the batch writer as follows:
with sleeper_client.create_batch_writer(table_name) as writer:
    # Create rows in a loop
    for _x in range(num_batches):
        rows = []
        for _y in range(1000):
            row = {"key": random_string(10, chars=string.digits), "value": random_string(30)}
            rows.append(row)

        # Write this batch to Sleeper
        writer.write(rows)